In [1]:
import numpy as np
import os,sys,glob
from pathlib import Path
import logging
from helper import (filterObjects,getModelInfo,saveOutput, \
                    electron_reco, muon_reco, deltaR, cutFlow, getD0,ee_acceptance,mm_acceptance,em_acceptance)
from numpy import ndarray
from typing import Any, Dict, List, Tuple, Union
import multiprocessing
import subprocess
from computeEfficiencies import getSR,preSelection,getObjects
DelphesLLP_path = Path(os.path.abspath("./DelphesLLP"))
os.environ['ROOT_INCLUDE_PATH'] = os.path.join(DelphesLLP_path,"external")

import ROOT
ROOT.gSystem.Load(os.path.join(DelphesLLP_path,"libDelphes.so"))
ROOT.gInterpreter.Declare('#include "classes/SortableObject.h"')
ROOT.gInterpreter.Declare('#include "classes/DelphesClasses.h"')
ROOT.gInterpreter.Declare('#include "external/ExRootAnalysis/ExRootTreeReader.h"')
from ROOT import TFile,Electron, Jet, MissingET, Muon, TTree

Welcome to JupyROOT 6.30/06


### Cutflow from ATLAS:

In [5]:
atlas_cutflow = 'ATLAS_data/HEPData-ins1831504-v2-csv/CutflowSR-ee.csv'
with open(atlas_cutflow) as f:
    lines = [l for l in f.readlines() if not l.startswith('#')]
cutFlowDict = {}
header = None
for l in lines:
    if not l.strip():
        continue
    if l.startswith(','):
        header = eval(l.split('=')[1].replace('GeV', '').replace('ns', '').replace('"', '').strip())
        cutFlowDict[header] = {}
    elif header is not None:
        cut, events = l.split(',')
        cutFlowDict[header][cut.strip()] = float(events.strip())

In [15]:
cFlow = cutFlowDict[(100,0.01)]
v0 = None
for k,val in cFlow.items():
    if v0 is None:
        v0 = val
        val_prev = val
    print(f"{k:<70}: {val/v0:1.3e} ({val/val_prev:1.3e})")
    val_prev = val

initial number of events ($\mathcal{L} \times \sigma$)                : 1.000e+00 (1.000e+00)
pass trigger and at least 2 baseline leptons                          : 1.448e-02 (1.448e-02)
2 leading leptons are electrons                                       : 7.732e-03 (5.340e-01)
$p_\text{T} > 65$ GeV                                                 : 6.492e-03 (8.397e-01)
$3$ mm$ < |d_{0}| < 300$ mm                                           : 2.380e-03 (3.667e-01)
both electrons pass isolation                                         : 2.302e-03 (9.669e-01)
$(p_\text{T}^\text{track}-p_\text{T}^e)/p_\text{T}^e$ $\geq -0.5$     : 1.672e-03 (7.265e-01)
ID track $\chi^2/n_{\mathrm{DOF}} <  2$                               : 1.517e-03 (9.071e-01)
number of missing layers $\leq 1$                                     : 1.517e-03 (1.000e+00)
$\Delta R_{\ell\ell}$ $> 0.2$                                         : 1.517e-03 (1.000e+00)
no additional cosmic muons                                  

In [7]:
inputFile = './pp2selsel/Events/run_09/selectron_400GeV_1.000ns_delphes_events.root'
f = TFile(inputFile,'read')
DelphesTree = f.Get('Delphes')
nevts = DelphesTree.GetEntries()
print(nevts)

30427


In [8]:
entry = 0
DelphesTree.GetEntry(entry)

3380

In [9]:
llps,muons,electrons = getObjects(DelphesTree)

In [10]:
for llp in llps:
    print(f"LLP: PT={llp.PT}, Eta={llp.Eta}, Phi={llp.Phi}, Mass={llp.Mass}, Charge={llp.Charge}, PID = {llp.PID}")

LLP: PT=280.3729553222656, Eta=-0.7693679928779602, Phi=2.6338045597076416, Mass=400.0, Charge=-1, PID = 1000011
LLP: PT=341.9228210449219, Eta=-0.46897053718566895, Phi=-0.4377497732639313, Mass=400.0, Charge=1, PID = -1000011


In [11]:
daughters = DelphesTree.bsmDirectDaughters
for d in daughters:
    print(f"Daughter: PT={d.PT}, Eta={d.Eta}, Phi={d.Phi},  Charge={d.Charge}, PID = {d.PID}")
    print(f'  D0: {getD0(d)}')

Daughter: PT=306.50048828125, Eta=-0.6262521743774414, Phi=2.047325372695923,  Charge=-1, PID = 11
  D0: 15.50494341339364
Daughter: PT=170.90524291992188, Eta=-0.1885974407196045, Phi=-2.2206625938415527,  Charge=0, PID = 1000039
  D0: 27.806480075269658
Daughter: PT=318.72515869140625, Eta=-0.5533028841018677, Phi=-0.9923588037490845,  Charge=1, PID = -11
  D0: 8.086581157233438
Daughter: PT=182.98321533203125, Eta=0.10164710879325867, Phi=0.7284526824951172,  Charge=0, PID = 1000039
  D0: 14.085428062106958


In [12]:
for el in electrons:
    reco_eff = electron_reco.efficiency(Lepton_p_textT_GeV=el.PT, Lepton_d_0_mm=abs(el.D0))
    print(f"Electron: PT={el.PT}, Eta={el.Eta}, Phi={el.Phi}, Charge={el.Charge}, PID = {el.PID}")
    print(f'  D0: {el.D0}, reco eff: {reco_eff}')

Electron: PT=306.50048828125, Eta=-0.6262521743774414, Phi=2.047325372695923, Charge=-1, PID = 11
  D0: 15.50494341339364, reco eff: 0.51151
Electron: PT=318.72515869140625, Eta=-0.5533028841018677, Phi=-0.9923588037490845, Charge=1, PID = -11
  D0: 8.086581157233438, reco eff: 0.51151


31.622776601683793